# Score All `S/T` Sites From FASTA

This notebook scores every serine/threonine site for each protein in the input FASTA file.

It follows the same resumable batch pattern used in `8.Sequence Binding Prediction for Morf.ipynb`, but writes a single site-level output CSV.


In [1]:
from pathlib import Path
import gzip

import pandas as pd
from Bio import SeqIO

from features import predict_all_sites


/home/luvul/.conda/envs/1433predictor2026/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FASTA_PATH = Path("/scratch/luvul_root/luvul0/luvul/proteomes/UP000005640_9606.fasta.gz")
OUTPUT_DIR = Path("/scratch/luvul_root/luvul0/luvul/proteomes/human_proteome_st_site_predictions")
BATCH_DIR = OUTPUT_DIR / "batches"
COMPLETED_BATCH_DIR = OUTPUT_DIR / "completed_rows"
FINAL_SITE_OUTPUT = OUTPUT_DIR / "human_proteome_st_site_scores.csv"

THRESHOLD = 0.58
BATCH_SIZE = 25
MAX_PROTEINS = None  # set to an integer for a short test run

SITE_OUTPUT_COLUMNS = [
    "row_id",
    "record_id",
    "uniprot_accession",
    "description",
    "sequence_length",
    "site",
    "residue",
    "prediction",
    "predicted_positive_model",
    "predicted_positive_threshold",
    "positive_probability",
    "negative_probability",
    "threshold",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(parents=True, exist_ok=True)
COMPLETED_BATCH_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def extract_accession(record_id):
    parts = record_id.split("|")
    return parts[1] if len(parts) >= 2 else record_id


def load_fasta_records(fasta_path):
    open_fn = gzip.open if fasta_path.suffix == ".gz" else open
    rows = []
    with open_fn(fasta_path, "rt") as handle:
        for row_id, record in enumerate(SeqIO.parse(handle, "fasta")):
            sequence = str(record.seq).strip().upper()
            rows.append(
                {
                    "row_id": int(row_id),
                    "record_id": record.id,
                    "uniprot_accession": extract_accession(record.id),
                    "description": record.description,
                    "sequence": sequence,
                    "sequence_length": int(len(sequence)),
                    "candidate_site_count": int(sum(residue in {"S", "T"} for residue in sequence)),
                }
            )
    fasta_df = pd.DataFrame(rows)
    if MAX_PROTEINS is not None:
        fasta_df = fasta_df.head(int(MAX_PROTEINS)).copy()
    return fasta_df


proteome_df = load_fasta_records(FASTA_PATH)
proteome_df.head()


,row_id,record_id,uniprot_accession,description,sequence,sequence_length,candidate_site_count
0,0,sp|A0A0J9YXV3|GREP1_HUMAN,A0A0J9YXV3,sp|A0A0J9YXV3|GREP1_HUMAN Glycine-rich extrace...,MGAWAFPAALFLLCLTSESLQGGLPLLPPGLGKVYGPHSGLGAGYD...,536,25
1,1,sp|A6NGW2|STRCL_HUMAN,A6NGW2,sp|A6NGW2|STRCL_HUMAN Putative stereocilin-lik...,MALSLWPLLLLLLLLLLLSFAVTLAPTGPHSLDPGLSFLKSLLSTL...,1772,211
2,2,sp|A6NIY4|SPDE5_HUMAN,A6NIY4,sp|A6NIY4|SPDE5_HUMAN Speedy protein E5 OS=Hom...,MDRTETRFRKRGQITEKITTSRQPQPQNEQSPQRSTSGYPLQEVVD...,402,38
3,3,sp|B3EWG6|FM25G_HUMAN,B3EWG6,sp|B3EWG6|FM25G_HUMAN Protein FAM25G OS=Homo s...,MLGGLGKLAAEGLAHRTEKATEGAIHAVEEVVKEVVGHAKETGEKA...,89,11
4,4,sp|C4AMC7|WASH3_HUMAN,C4AMC7,sp|C4AMC7|WASH3_HUMAN Putative WAS protein fam...,MTPVRMQHSLAGQTYAVPLIQPDLRREEAVQQMADALQYLQKVSGD...,463,56


## Quick Test Block

Run this block first to verify that a few proteins from the FASTA can be scored before starting the full batch job.


In [4]:
TEST_PROTEIN_COUNT = 3
TEST_SITES_TO_SHOW_PER_PROTEIN = 10

test_df = proteome_df[proteome_df["candidate_site_count"] > 0].head(TEST_PROTEIN_COUNT).copy()
test_summary_rows = []
test_site_rows = []

for _, row in test_df.iterrows():
    try:
        site_results_df, _, _ = predict_all_sites(row["sequence"])
        ranked_df = site_results_df.sort_values(["positive_probability", "site"], ascending=[False, True]).reset_index(drop=True)
        best_row = ranked_df.iloc[0]

        test_summary_rows.append(
            {
                "row_id": int(row["row_id"]),
                "uniprot_accession": row["uniprot_accession"],
                "sequence_length": int(row["sequence_length"]),
                "candidate_site_count": int(row["candidate_site_count"]),
                "scored_site_count": int(len(site_results_df)),
                "best_site": int(best_row["site"]),
                "best_residue": best_row["residue"],
                "best_site_probability": float(best_row["positive_probability"]),
                "test_status": "ok",
                "error_message": "",
            }
        )

        preview_df = ranked_df.head(TEST_SITES_TO_SHOW_PER_PROTEIN).copy()
        for preview_row in preview_df.itertuples():
            test_site_rows.append(
                {
                    "row_id": int(row["row_id"]),
                    "uniprot_accession": row["uniprot_accession"],
                    "site": int(preview_row.site),
                    "residue": preview_row.residue,
                    "prediction": preview_row.prediction,
                    "predicted_positive_model": bool(preview_row.predicted_positive),
                    "positive_probability": float(preview_row.positive_probability),
                    "negative_probability": float(preview_row.negative_probability),
                }
            )
    except Exception as exc:
        test_summary_rows.append(
            {
                "row_id": int(row["row_id"]),
                "uniprot_accession": row["uniprot_accession"],
                "sequence_length": int(row["sequence_length"]),
                "candidate_site_count": int(row["candidate_site_count"]),
                "scored_site_count": None,
                "best_site": None,
                "best_residue": None,
                "best_site_probability": None,
                "test_status": "error",
                "error_message": str(exc),
            }
        )

test_results_df = pd.DataFrame(test_summary_rows)
test_site_preview_df = pd.DataFrame(test_site_rows)

print(f"Tested {len(test_results_df)} proteins from {FASTA_PATH.name}")
print(f"Showing up to {TEST_SITES_TO_SHOW_PER_PROTEIN} scored sites per test protein")
test_results_df


Tested 3 proteins from UP000005640_9606.fasta.gz
Showing up to 10 scored sites per test protein


,row_id,uniprot_accession,sequence_length,candidate_site_count,scored_site_count,best_site,best_residue,best_site_probability,test_status,error_message
0,0,A0A0J9YXV3,536,25,25,485,T,0.442039,ok,
1,1,A6NGW2,1772,211,211,347,T,0.762236,ok,
2,2,A6NIY4,402,38,38,37,S,0.914991,ok,


In [5]:
test_site_preview_df


,row_id,uniprot_accession,site,residue,prediction,predicted_positive_model,positive_probability,negative_probability
0,0,A0A0J9YXV3,485,T,0,False,0.442039,0.557961
1,0,A0A0J9YXV3,354,T,0,False,0.299345,0.700655
2,0,A0A0J9YXV3,369,S,0,False,0.161809,0.838191
3,0,A0A0J9YXV3,107,S,0,False,0.100751,0.899249
4,0,A0A0J9YXV3,327,S,0,False,0.097169,0.902831
5,0,A0A0J9YXV3,321,T,0,False,0.092500,0.907500
6,0,A0A0J9YXV3,223,T,0,False,0.078266,0.921734
7,0,A0A0J9YXV3,19,S,0,False,0.075318,0.924682
8,0,A0A0J9YXV3,181,S,0,False,0.074784,0.925216
9,0,A0A0J9YXV3,64,T,0,False,0.067753,0.932247


In [ ]:
def list_batch_files(batch_dir):
    return sorted(batch_dir.glob("batch_*.csv"))


def load_processed_row_ids(batch_dir, completed_batch_dir):
    processed = set()
    for search_dir in [completed_batch_dir, batch_dir]:
        for batch_file in list_batch_files(search_dir):
            batch_df = pd.read_csv(batch_file, usecols=["row_id"])
            processed.update(batch_df["row_id"].astype(int).tolist())
    return processed


def save_batch(batch_df, batch_index, batch_dir):
    batch_path = batch_dir / f"batch_{batch_index:04d}.csv"
    batch_df.to_csv(batch_path, index=False)
    return batch_path


def save_completed_rows(batch_input_df, batch_index, completed_batch_dir):
    completed_path = completed_batch_dir / f"batch_{batch_index:04d}.csv"
    batch_input_df[["row_id"]].drop_duplicates().to_csv(completed_path, index=False)
    return completed_path


def combine_saved_batches(batch_dir, final_output, sort_columns):
    batch_files = list_batch_files(batch_dir)
    if not batch_files:
        return pd.DataFrame(columns=SITE_OUTPUT_COLUMNS)

    combined_df = pd.concat((pd.read_csv(batch_file) for batch_file in batch_files), ignore_index=True)
    combined_df = combined_df.sort_values(sort_columns).reset_index(drop=True)
    combined_df.to_csv(final_output, index=False)
    return combined_df


def predict_protein_sites(row, threshold):
    row_id = int(row["row_id"])
    sequence = row["sequence"].strip()

    if not sequence:
        return []

    if int(row["candidate_site_count"]) == 0:
        return []

    try:
        site_results_df, _, _ = predict_all_sites(sequence)
        site_results_df = site_results_df.sort_values(["site"]).reset_index(drop=True)
        site_results_df["predicted_positive_threshold"] = (
            site_results_df["positive_probability"] >= float(threshold)
        )

        site_rows = []
        for site_row in site_results_df.itertuples():
            site_rows.append(
                {
                    "row_id": row_id,
                    "record_id": row["record_id"],
                    "uniprot_accession": row["uniprot_accession"],
                    "description": row["description"],
                    "sequence_length": int(row["sequence_length"]),
                    "site": int(site_row.site),
                    "residue": site_row.residue,
                    "prediction": site_row.prediction,
                    "predicted_positive_model": bool(site_row.predicted_positive),
                    "predicted_positive_threshold": bool(site_row.predicted_positive_threshold),
                    "positive_probability": float(site_row.positive_probability),
                    "negative_probability": float(site_row.negative_probability),
                    "threshold": float(threshold),
                }
            )

        return site_rows
    except Exception as exc:
        print(f"Error scoring {row['record_id']}: {exc}")
        return []


In [ ]:
processed_row_ids = load_processed_row_ids(BATCH_DIR, COMPLETED_BATCH_DIR)
remaining_df = proteome_df[~proteome_df["row_id"].isin(processed_row_ids)].copy()

print(f"Total proteins: {len(proteome_df)}")
print(f"Already processed: {len(processed_row_ids)}")
print(f"Remaining: {len(remaining_df)}")
print(f"Threshold: {THRESHOLD}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Final site output: {FINAL_SITE_OUTPUT}")


In [ ]:
if len(remaining_df) == 0:
    print("All proteins are already processed.")
else:
    for batch_index, start in enumerate(range(0, len(proteome_df), BATCH_SIZE), start=1):
        batch_path = BATCH_DIR / f"batch_{batch_index:04d}.csv"
        completed_path = COMPLETED_BATCH_DIR / f"batch_{batch_index:04d}.csv"

        if batch_path.exists() or completed_path.exists():
            print(f"Skipping {batch_path.name}: existing chunk file detected.")
            continue

        batch_input_df = proteome_df.iloc[start : start + BATCH_SIZE].copy()
        batch_input_df = batch_input_df[~batch_input_df["row_id"].isin(processed_row_ids)].copy()

        if batch_input_df.empty:
            print(f"Skipping {batch_path.name}: all rows in this chunk are already processed.")
            continue

        site_results = []
        for _, row in batch_input_df.iterrows():
            site_rows = predict_protein_sites(row, THRESHOLD)
            site_results.extend(site_rows)

        site_batch_df = pd.DataFrame(site_results, columns=SITE_OUTPUT_COLUMNS)
        batch_path = save_batch(site_batch_df, batch_index, BATCH_DIR)
        completed_path = save_completed_rows(batch_input_df, batch_index, COMPLETED_BATCH_DIR)
        processed_row_ids.update(batch_input_df["row_id"].astype(int).tolist())

        print(
            f"Saved {batch_path.name} and {completed_path.name}: "
            f"rows {int(batch_input_df['row_id'].min())} to {int(batch_input_df['row_id'].max())}"
        )


In [ ]:
final_site_df = combine_saved_batches(
    BATCH_DIR,
    FINAL_SITE_OUTPUT,
    sort_columns=["row_id", "site"],
)

print(f"Site-level scores saved to: {FINAL_SITE_OUTPUT}")
print(f"Site rows in final output: {len(final_site_df)}")

final_site_df.head()


## Notes

- Re-running the notebook resumes from `batches` and `completed_rows`.
- `THRESHOLD` is only used to label `predicted_positive_threshold`; the raw `positive_probability` score is always saved.
- The main site-level output is `human_proteome_st_site_scores.csv`.
